# 13.10 偏好比較能教另一個評分模型嗎？


對「只回數字：1+2=?」，我們標「3」比「1 加 2 是 3。」合適。現在接收這個比較的，不是未來回答的模型，而是另一個專門給分的模型。它叫**獎勵模型（reward model）**，讀問題與回答，輸出估計這份回答有多合適的分數。

先用手設分數看學習目標。較佳分數減較差分數為d，將`sigmoid(d)=1/(1+exp(−d))`當成較佳勝出的估計機率，再取負log作代價。差0表示0.5；差2約0.8808。這是偏好選擇的估計，不是內容正確率。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch
from tiny_perceptron.posttraining import preference_loss

for preferred_score in [0.0, 2.0]:
    chosen = torch.tensor([preferred_score])
    rejected = torch.tensor([0.0])
    gap = chosen - rejected
    win_probability = torch.sigmoid(gap)
    loss = preference_loss(chosen, rejected)
    print("差距", gap.item(), "勝出機率", round(win_probability.item(), 4), "代價", round(loss.item(), 4))

差距 0.0 勝出機率 0.5 代價 0.6931
差距 2.0 勝出機率 0.8808 代價 0.1269


輸出兩行機率與代價約為`0.5、0.6931`和`0.8808、0.1269`。把較佳相對分數拉高，代價就降低；兩個分數一起加10，差距與代價不變。因此原始分數3不代表三成正確，也不是比1多懂三倍。

接下來PPO用一個有限選卡任務，把評分和回答選擇分開。以1+2題為例，候選先由Python寫好：

| 卡號 | 加法候選文字 |
| ---: | --- |
| 0 | 3 |
| 1 | 1 加 2 是 3。 |
| 2 | 4 |
| 3 | 請再提供更多資訊。 |

只回數字時選0，一句話說明時選1。另有「單價1元、運費2元，總價多少？」的題，因缺購買數量，應選求補數量的3號卡，不能擅自把單價加運費當總價。

小網路讀的是已整理好的題目條件與卡號特徵，不讀中文；候選裡的加法也先算好了。偏好標籤由作者規則產生，沒有招募真人，所以它只教流程，不是完整人類回饋RLHF實證。

練習把rejected分數改1，兩個差距變−1與1，機率約0.2689與0.7311。再把雙方都加10，核對結果不變。

<details>
<summary>補充：來源、完整設定與既有實測紀錄</summary>

實際訓練時先用訓練家族的偏好對更新評分網路，再凍結它，供[13.15](https://birdhackor.github.io/tiny-perceptron-vlm/13.15.html)的選擇模型使用。數字對調仍屬同一家族，不會把`1+2`與`2+1`分到訓練和最後測試兩邊。評分員是否把未見家族也排對，與它的訓練代價，會分別保存；預寫卡裡有正確答案，不等於模型會自己寫出答案。

</details>
